In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns


import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
from celloracle import motif_analysis as ma

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


import seaborn as sns

import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
from celloracle import motif_analysis as ma
from celloracle.utility import save_as_pickled_object
co.__version__

/home/zhanglab/mambaforge/envs/celloracle/lib/python3.9/site-packages/loompy/bus_file.py:68: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def twobit_to_dna(twobit: int, size: int) -> str:
/home/zhanglab/mambaforge/envs/celloracle/lib/python3.9/site-packages/loompy/bus_file.py:85: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def dna_to_twobit(dna: str) -> int:


1 package does not meet CellOracle requirement.
 Your igraph version is 0.9.11. Please install igraph>=0.10.1


'0.14.0'

In [6]:
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = (15,7)
plt.rcParams["savefig.dpi"] = 600

In [7]:
# PLEASE make sure reference genome is correct.
ref_genome = "mm10"

genome_installation = ma.is_genome_installed(ref_genome=ref_genome,
                                             genomes_dir=None)
print(ref_genome, "installation: ", genome_installation)

mm10 installation:  True


In [3]:
# Load scATAC-seq peak list.
peaks = pd.read_csv("../process/regulation/coaccessibility/celloracle_prepare_all_peaks.csv", index_col=0)
peaks = peaks.x.values
peaks

array(['chr1:3094550-3095050', 'chr1:3268257-3268757',
       'chr1:3395775-3396275', ..., 'chrX:169950414-169950914',
       'chrX:169950989-169951489', 'chrX:169960677-169961177'],
      dtype=object)

In [4]:
def convert_intervals(intervals):
    output_intervals = []
    for interval in intervals:
        chrom, coords = interval.split(':')
        start, end = coords.split('-')
        output_interval = f"{chrom}_{start}_{end}"
        output_intervals.append(output_interval)
    return output_intervals

peaks = convert_intervals(peaks)


In [6]:
# Load Cicero coaccessibility scores.
cicero_connections = pd.read_csv("../process/regulation/coaccessibility/20241213_archR_ca_celloracle_prepare.csv", index_col=0)
cicero_connections.head()

,Peak1,Peak2,coaccess
1,chr1:3395775-3396275,chr1:3398879-3399379,0.411792
2,chr1:3395775-3396275,chr1:3399772-3400272,0.254039
3,chr1:3398879-3399379,chr1:3395775-3396275,0.411792
4,chr1:3399772-3400272,chr1:3395775-3396275,0.254039
5,chr1:3547915-3548415,chr1:3549186-3549686,0.279565


In [7]:
##!! Please make sure to specify the correct reference genome here
tss_annotated = ma.get_tss_info(peak_str_list=peaks, ref_genome="mm10")

# Check results
tss_annotated.tail()

que bed peaks: 210111
tss peaks in que: 27858


,chr,start,end,gene_short_name,strand
27853,chr12,100899033,100899533,Gpr68,-
27854,chr12,100900511,100901011,Gpr68,-
27855,chr4,129490925,129491425,Fam229a,-
27856,chr4,129491455,129491955,Fam229a,-
27857,chr17,24473754,24474254,Bricd5,+


In [8]:
cicero_connections["Peak1"] = convert_intervals(cicero_connections["Peak1"] )

In [10]:
cicero_connections["Peak2"] = convert_intervals(cicero_connections["Peak2"] )

In [11]:
integrated = ma.integrate_tss_peak_with_cicero(tss_peak=tss_annotated,
                                               cicero_connections=cicero_connections)
print(integrated.shape)
integrated.head()

(86314, 3)


,peak_id,gene_short_name,coaccess
0,chr10_100015477_100015977,Kitl,1.000000
1,chr10_100025659_100026159,Kitl,0.239613
2,chr10_100072501_100073001,Kitl,0.288837
3,chr10_100073073_100073573,Kitl,0.310684
4,chr10_100487197_100487697,Tmtc3,1.000000


In [14]:
peak = integrated[integrated.coaccess >= 0.7]
peak = peak[["peak_id", "gene_short_name"]].reset_index(drop=True)

In [15]:
print(peak.shape)
peak.head()

(26723, 2)


,peak_id,gene_short_name
0,chr10_100015477_100015977,Kitl
1,chr10_100487197_100487697,Tmtc3
2,chr10_100487787_100488287,Tmtc3
3,chr10_100588978_100589478,4930430F08Rik
4,chr10_100741985_100742485,Gm35722


In [16]:
peak.to_csv("../process/regulation/celloracle/20241213_processed_peak.csv")

In [18]:
# PLEASE make sure reference genome is correct.
ref_genome = "mm10"

genome_installation = ma.is_genome_installed(ref_genome=ref_genome,
                                             genomes_dir=None)
print(ref_genome, "installation: ", genome_installation)

mm10 installation:  True


In [20]:
peak.head()

,peak_id,gene_short_name
0,chr10_100015477_100015977,Kitl
1,chr10_100487197_100487697,Tmtc3
2,chr10_100487787_100488287,Tmtc3
3,chr10_100588978_100589478,4930430F08Rik
4,chr10_100741985_100742485,Gm35722


In [25]:
genome_installation = ma.is_genome_installed(ref_genome=ref_genome,
                                             genomes_dir=None)
genome_installation

True

In [26]:
# Load annotated peak data.
peaksDf = pd.read_csv("../process/regulation/celloracle/20241213_processed_peak.csv", index_col=0)

In [27]:
peaksDf = ma.check_peak_format(peaks_df=peaksDf, ref_genome=ref_genome, genomes_dir=None)

Peaks before filtering:  26723
Peaks with invalid chr_name:  0
Peaks with invalid length:  0
Peaks after filtering:  26723


In [28]:
# Instantiate TFinfo object
tfi = ma.TFinfo(peak_data_frame=peaksDf,
                ref_genome=ref_genome,
                genomes_dir=None)

In [29]:
%%time
# Scan motifs. !!CAUTION!! This step may take several hours if you have many peaks!
tfi.scan(fpr=0.02,
         motifs=None,  # If you enter None, default motifs will be loaded.
         verbose=True)

# Save tfinfo object
tfi.to_hdf5(file_path="../process/regulation/celloracle/20241213.celloracle.tfinfo")

No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 

Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 

Motif scan started .. It may take long time.



scanning:   0%|          | 0/25418 [00:00<?, ? sequences/s]

CPU times: user 23min 47s, sys: 4min 20s, total: 28min 7s
Wall time: 28min 31s


In [30]:
# Reset filtering
tfi.reset_filtering()

# Do filtering
tfi.filter_motifs_by_score(threshold=10)

# Format post-filtering results.
tfi.make_TFinfo_dataframe_and_dictionary(verbose=True)

Filtering finished: 7479866 -> 1584790
1. Converting scanned results into one-hot encoded dataframe.


  0%|          | 0/25418 [00:00<?, ?it/s]

2. Converting results into dictionaries.


  0%|          | 0/16164 [00:00<?, ?it/s]

  0%|          | 0/1094 [00:00<?, ?it/s]

In [31]:
df = tfi.to_dataframe()
df.head()

,peak_id,gene_short_name,9430076c15rik,Ac002126.6,Ac012531.1,Ac226150.2,Afp,Ahr,Ahrr,Aire,...,Znf784,Znf8,Znf816,Znf85,Zscan10,Zscan16,Zscan22,Zscan26,Zscan31,Zscan4
0,chr10_100015477_100015977,Kitl,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,chr10_100487197_100487697,Tmtc3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,chr10_100487787_100488287,Tmtc3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,chr10_100588978_100589478,4930430F08Rik,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,chr10_100741985_100742485,Gm35722,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
df.shape

(26723, 1096)

In [39]:
df.columns

Index(['peak_id', 'gene_short_name', '9430076c15rik', 'Ac002126.6',
       'Ac012531.1', 'Ac226150.2', 'Afp', 'Ahr', 'Ahrr', 'Aire',
       ...
       'Znf784', 'Znf8', 'Znf816', 'Znf85', 'Zscan10', 'Zscan16', 'Zscan22',
       'Zscan26', 'Zscan31', 'Zscan4'],
      dtype='object', length=1096)

In [50]:
Arid3a_target = pd.DataFrame(df.loc[:,["gene_short_name","peak_id"]][df.loc[:,"Arid3a"] != 0])

In [52]:
Arid3a_target.to_csv("../process/regulation/celloracle/20241213_arid3a_gene.csv")

In [40]:
"Arid3a" in df.columns

True

In [36]:
len(df.loc[:,"gene_short_name"].unique())

16164

In [53]:
df.to_parquet("../process/regulation/celloracle/20241213_base_GRN_dataframe.parquet")